In [ ]:
from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes, TR_attributes
import pandas as pd
import numpy as np
from Utilities.excel_loaders import conn_out_xload
from Utilities.dfutils import dict_iloc
from Utilities.Storage import get_curr_storage_path
from Utilities.func_utils import load_arguments
import pickle
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import datetime as dt
import seaborn as sns
import mpld3
import matplotlib.dates as mdates

In [ ]:
STORAGE_ = get_curr_storage_path()
l_path = '//192.168.10.91/data/Data/orderbooks/base/'

def load_ob(m, t, dt, p_d, bT, eT):
    ob_class = OrderBookSnaps(verbose=True)
    file_path = l_path + t.split('_')[0] + '/'
    file_name = m + '_' + t.split('_')[0] + '_' + p_d.strftime('%y%m%d') + '_' + dt.strftime('%y%m%d') + '.p'
    print('%s Loading OrderBook %d...' % (dt.strftime('%y-%m-%d'), 0))
    print(file_path + file_name)
    time_load = ob_class.import_data(file_path + file_name)
    print('OrderBook %d created in %d sec' % (0, time_load))
    # ob_class.LoB_truncate(thres_vol=1)
    return ob_class.LoB_select(bT, eT, freq=None)

def variables_from_instrument(instrument: str):
    result = {
        'mkt': None,
        'tenor': None,
        'tn': None
    }
    for x in ['de', 'fr', 'ttf']:
        if x in instrument:
            result['mkt'] = x
    result['tenor'] = instrument[-2]
    result['tn'] = int(instrument[-1])
    return result

In [ ]:
dates_out = conn_out_xload()
allwd_broker_ids = [1441]

# ------------------ dataset prep ---------------------------------

# Load arguments
_INSTRUMENTS = ['dey1']
_START_DATE, _END_DATE = '2024-11-01', '2025-04-01'
ins_dicts = [variables_from_instrument(x) for x in _INSTRUMENTS]
n_s = 2
mkt_list = [ins_dict['mkt'] for ins_dict in ins_dicts]
tenor_list = [ins_dict['tenor'] for ins_dict in ins_dicts]
tn1_list = [ins_dict['tn'] for ins_dict in ins_dicts]
ts_lag = (lambda i: mkt_list[i] + tenor_list[i] + str(tn1_list[i]))(0)

tn2_list = []
prod = 'base'
venue_list = ['eex']
start_date = datetime.strptime(_START_DATE, '%Y-%m-%d').date()
end_date = datetime.strptime(_END_DATE, '%Y-%m-%d').date()

if not tn2_list:
    tn_list = [str(t1) for t1 in tn1_list]
else:
    tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]

dates = pd.date_range(start_date, end_date, freq='B')

spread_class = SpreadSingle(mkt_list, tenor_list, tn1_list, tn2_list, venue_list)
product_date1 = spread_class.product_dates(dates, n_s, tn_bool=True)
product_date2 = spread_class.product_dates(dates, n_s, tn_bool=False)

start_time = time(9, 0, 0, 0)
end_time = time(17, 40, 0, 0)

gran = None

is_db = True
is_tr = True

if is_db:
    data_class = TPData() 
else:
    data_class = TPDataDa()



data_class.create_connection('OracleSQL')
inst_trades = data_class.get_trades_inst('de', venue_list, start_date, end_date, prod='base', spread_bool=False)
instrument_ts = inst_trades[inst_trades.eval('broker_id==1441')].index.drop_duplicates()


obAtt = OB_attributes(['b_price', 'a_price'])
df_orders = {}
df_trades = {}

for k, ds in enumerate(dates):
    if ds in dates_out:
        continue
    try:
        bT = datetime.combine(ds, start_time)
        eT = datetime.combine(ds, end_time)
        pd1_aux = [None if p is None else p[k] for p in product_date1]
        pd2_aux = [None if p is None else p[k] for p in product_date2]
        for (m, t, n, pd1, pd2) in zip(mkt_list, tenor_list, tn_list,
                                                    pd1_aux, pd2_aux):
            i = m + t + str(n)
            # Order Book attributes
            ob_class = OrderBookSnaps(verbose=True)
            LoB, ts = load_ob(m, t, bT, pd1, bT, eT)
            ob_class.update_data(LoB, ts)
            orders = obAtt.prepare_ob_data(LoB, [0], aonn=True)
            # Trades
            data_class.create_connection('OracleSQL')
            trades = data_class.get_trades(m, t, venue_list, pd1, bT, eT,
                                                prod)
            trades = trades[trades['broker_id'].isin(allwd_broker_ids)]
            trades = trades.reset_index(names='datetime').drop_duplicates('datetime', keep='last').set_index('datetime')
            trades = data_class.clean_trades(trades, pd.DataFrame(orders), is_verbose=True)

            if i not in df_trades:
                df_trades[i] = trades
                df_orders[i] = pd.DataFrame(orders).set_index('timestamp')
            else:
                df_trades[i] = pd.concat([df_trades[i], trades])
                df_orders[i] = pd.concat([df_orders[i], pd.DataFrame(orders).set_index('timestamp')])
    except:
        continue

Connected to the database oracle
Disconnected from the database oracle
24-11-01 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_250101_241101.p
OrderBook 0 created in 19 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_id, own_trades]
Index: []


24-11-04 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_250101_241104.p
OrderBook 0 created in 17 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_id, own_trades]
Index: []


24-11-05 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_250101_241105.p
OrderBook 0 created in 15 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_

C:\Users\krajcovic\Documents\GitHub\EnergyTrading\Python\Database\TPData.py:556: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_out.index = df_out.index + pd.to_timedelta(df_data.loc[~idx, 'nanotime'], unit='ns')


OrderBook 0 created in 13 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_id, own_trades]
Index: []


25-01-03 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250103.p
OrderBook 0 created in 18 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_id, own_trades]
Index: []


25-01-06 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250106.p
OrderBook 0 created in 21 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [tradeid, price, volume, action, broker_id, own_trades]
Index: []


25-01-07 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250107.p
OrderBook 0 created in 18 sec
Connected to 

In [ ]:
file_path = r'C:\Users\krajcovic\Documents\Trading\test_data\dey1_202500201_20250401.pkt'
df_trades['dey1'].to_parquet(file_path)
df = df_trades['dey1'].loc[dt.datetime(2025,1,1):].copy()

In [ ]:
file_path = r'C:\Users\krajcovic\Documents\Trading\test_data\dey1_202500201_20250401.pkt'
df = pd.read_parquet(file_path).sort_index().loc[dt.datetime(2025,1,1):].copy()

In [ ]:
# Function to generate OHLC candles from price series
def generate_candles(df, granularity='5T'):
    ohlc_dict = {
        'price': ['first', 'max', 'min', 'last']
    }
    candles = df.resample(granularity).apply(ohlc_dict)
    candles.columns = ['open', 'high', 'low', 'close']
    return candles

In [ ]:
import pandas as pd

def calculate_MACD(df_lag,
                   price_col: str = 'close',
                   high_col: str = 'high',
                   low_col: str = 'low',
                   se: int = 12,
                   le: int = 26,
                   cont: int = 9) -> pd.DataFrame:
    """
    Compute ATR, MACD, ATR‐normalized MACD, and their 100-period percentile ranks.
    
    Returns a DataFrame indexed by datetime with columns:
      ['ATR', 'MACD', 'Signal', 'MACD_hist',
       'MACD_norm', 'MACD_hist_norm',
       'MACD_norm_pr', 'MACD_hist_norm_pr']
    """
    df = df_lag.copy()
    # ensure datetime column
    if 'datetime' not in df.columns:
        df = df.reset_index().rename(columns={'index':'datetime'})
    df = df.dropna(subset=[price_col, high_col, low_col])
    
    # True Range
    hl = df[high_col] - df[low_col]
    hc = (df[high_col] - df[price_col].shift()).abs()
    lc = (df[low_col]  - df[price_col].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    # ATR with Wilder’s smoothing
    atr = tr.ewm(alpha=1/se, adjust=False, min_periods=se).mean()
    
    # MACD
    close = df[price_col]
    short_ema = close.ewm(span=se, adjust=False).mean()
    long_ema  = close.ewm(span=le, adjust=False).mean()
    macd      = short_ema-long_ema
    signal    = macd.ewm(span=cont, adjust=False).mean()
    hist      = macd - signal
    
    # Normalize by ATR
    macd_norm = macd / atr
    hist_norm = hist / atr
    
    # 100-period percentile rank (0–100)
    macd_pr = macd_norm.rolling(window=100, min_periods=100)\
        .apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)
    hist_pr = hist_norm.rolling(window=100, min_periods=100)\
        .apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)
    
    # assemble output
    out = pd.DataFrame({
        'ATR':               atr,
        'MACD':              macd,
        'Signal':            signal,
        'MACD_hist':         hist,
        'MACD_norm':         macd_norm,
        'MACD_hist_norm':    hist_norm,
        'MACD_norm_pr':      macd_pr,
        'MACD_hist_norm_pr': hist_pr
    })
    out.index = df['datetime']
    out.index.name = 'datetime'
    return out


In [ ]:
candles = generate_candles(df, granularity='1h').dropna()
macd_df = calculate_MACD(candles).dropna()
candles = pd.concat([candles, macd_df],axis=1)

In [ ]:
# High/Lows code
i = 0
# Initialize variables
last_low, last_high = np.inf, -np.inf  # Proper extreme values
highs, lows = [], []
 
# Iterate through OHLC data
for i, row in enumerate(ohlc_5m.dropna().to_dict(orient='records')):
    o, h, l, c = row['open'], row['high'], row['low'], row['close']
    color = 'g' if c > o else 'r'
    value_dict = lambda t, v, idx, col: {'type': t, 'value': v, 'index': idx, 'color': col}
 
    # Update highs
    if h > last_high:
        if not highs or last_high < highs[-1]['value']:
            highs.append(value_dict('h', h, i, color))
        else:
            highs[-1] = value_dict('h', h, i, color)
 
    # Update lows
    if l < last_low:
        if not lows or last_low > lows[-1]['value']:
            lows.append(value_dict('l', l, i, color))
        else:
            lows[-1] = value_dict('l', l, i, color)
 
    # Update last high and low
    last_high, last_low = h, l

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.widgets import MultiCursor

# choose which columns to plot
upper_col = 'MACD_norm'
lower_col = 'MACD_hist'  # replace with your actual column name

fig, (ax1, ax2, ax3) = plt.subplots(
    nrows=3,
    ncols=1,
    sharex=True,
    figsize=(10, 10),
    gridspec_kw={'height_ratios': [3, 1, 1]}
)

# integer x-coords
x = np.arange(len(candles))
width = 0.6  # 60% of slot

# ─── Top: candlesticks ───────────────────────────────────────────────────────
for i, (ts, row) in enumerate(candles.iterrows()):
    if pd.notna(row.open) and pd.notna(row.close):
        ax1.vlines(i, row.low, row.high, color='black', linewidth=1)
        ax1.bar(
            i,
            row.close - row.open,
            width,
            bottom=row.open,
            color='green' if row.close > row.open else 'red',
            edgecolor='black',
            align='center'
        )
ax1.set_ylabel('Price')
ax1.set_title('Price Candles')
# autoscale Y tightly so it doesn't include zero
min_low  = candles['low'].min()
max_high = candles['high'].max()
pad      = (max_high - min_low) * 0.05
ax1.set_ylim(min_low - pad, max_high + pad)
ax1.grid(True)  # add grid

# ─── Middle: first selected series ───────────────────────────────────────────
ax2.plot(x, candles[upper_col], marker='o', linestyle='-')
ax2.set_ylabel(upper_col)
ax2.grid(True)  # add grid

# ─── Bottom: new selected series ────────────────────────────────────────────
ax3.plot(x, candles[lower_col], marker='o', linestyle='-')
ax3.set_ylabel(lower_col)
ax3.grid(True)  # add grid
ax3.set_xlabel('Date')

# ─── X-ticks & labels on the bottom axis only ──────────────────────────────
N = max(len(candles)//10, 1)
tick_positions = x[::N]
tick_labels    = candles.index.strftime('%Y-%m-%d %H:%M')[::N]
ax3.set_xticks(tick_positions)
ax3.set_xticklabels(tick_labels, rotation=45, ha='right')

# ─── Custom hover/readout on the top plot ───────────────────────────────────
def fmt_coord(xi, y):
    idx = int(np.round(xi))
    if 0 <= idx < len(candles):
        dt = candles.index[idx]
        return f"{dt:%Y-%m-%d %H:%M}, {y:.2f}"
    return f"{y:.2f}"
ax1.format_coord = fmt_coord

# ─── Synchronized vertical crosshair on all three axes ────────────────────
multi = MultiCursor(
    fig.canvas,
    (ax1, ax2, ax3),
    horizOn=False,
    vertOn=True,
    color='gray',
    lw=1
)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

class MultiGranularityIndicator:
    """
    Builds indicators on multiple candle granularities: MACD on a finer granularity,
    and swing highs/lows on a coarser granularity, then combines into one DataFrame.
    """
    def __init__(self,
                 small_granularity: str = '5T',
                 large_granularity: str = '1H',
                 se: int = 12,
                 le: int = 26,
                 cont: int = 9):
        self.small_granularity = small_granularity
        self.large_granularity = large_granularity
        self.se = se
        self.le = le
        self.cont = cont

    @staticmethod
    def generate_candles(df: pd.DataFrame,
                         granularity: str) -> pd.DataFrame:
        """
        Resample a price series into OHLC candles.
        Expects a time-indexed DataFrame or Series named 'price'.
        Returns columns ['open','high','low','close'].
        """
        if isinstance(df, pd.Series):
            df = df.to_frame(name='price')
        ohlc = df.resample(granularity).agg({'price': ['first','max','min','last']})
        ohlc.columns = ['open','high','low','close']
        return ohlc

    @staticmethod
    def calculate_MACD(df_lag: pd.DataFrame,
                       price_col: str = 'close',
                       high_col: str = 'high',
                       low_col: str = 'low',
                       se: int = 12,
                       le: int = 26,
                       cont: int = 9) -> pd.DataFrame:
        """
        Compute ATR, MACD, ATR-normalized MACD, and 100-period percentile ranks.
        df_lag must have columns ['datetime', high_col, low_col, price_col].
        Returns DataFrame indexed by datetime with MACD & ATR stats.
        """
        df = df_lag.copy()
        if 'datetime' not in df.columns:
            df = df.reset_index().rename(columns={'index':'datetime'})
        df = df.dropna(subset=[price_col, high_col, low_col])

        # True Range & ATR
        hl = df[high_col] - df[low_col]
        hc = (df[high_col] - df[price_col].shift()).abs()
        lc = (df[low_col] - df[price_col].shift()).abs()
        tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/se, adjust=False, min_periods=se).mean()

        # MACD
        close = df[price_col]
        short_ema = close.ewm(span=se, adjust=False).mean()
        long_ema  = close.ewm(span=le, adjust=False).mean()
        macd      = short_ema - long_ema
        signal    = macd.ewm(span=cont, adjust=False).mean()
        hist      = macd - signal

        # Normalize
        macd_norm = macd / atr
        hist_norm = hist / atr

        # Percentile ranks
        macd_pr = macd_norm.rolling(window=100, min_periods=100)
        macd_pr = macd_pr.apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)
        hist_pr = hist_norm.rolling(window=100, min_periods=100)
        hist_pr = hist_pr.apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)

        out = pd.DataFrame({
            'ATR':               atr,
            'MACD':              macd,
            'Signal':            signal,
            'MACD_hist':         hist,
            'MACD_norm':         macd_norm,
            'MACD_hist_norm':    hist_norm,
            'MACD_norm_pr':      macd_pr,
            'MACD_hist_norm_pr': hist_pr
        })
        out.index = pd.to_datetime(df['datetime'])
        out.index.name = 'datetime'
        return out

    @staticmethod
    def detect_swing_points(ohlc_df: pd.DataFrame,
                            high_col: str = 'high',
                            low_col: str = 'low') -> pd.DataFrame:
        """
        Compute swing highs/lows based on the posted algorithm. Marks new extremes and recalculates opposite swing.
        Returns DataFrame with 'swing_high' and 'swing_low' columns.
        """
        df = ohlc_df.copy().reset_index()
        df['min'] = np.nan
        df['max'] = np.nan
        for idx, row in df.iterrows():
            o, h, l, c = row['open'], row['high'], row['low'], row['close']
            if idx == 0:
                last_min = [idx, l]
                last_min2 = [idx, l]
                last_max = [idx, h]
                last_max2 = [idx, h]
                df.at[idx, 'min'] = l
                df.at[idx, 'max'] = h
                continue
            last_max_idx_diff = idx - last_max[0]
            last_min_idx_diff = idx - last_min[0]
            if last_max[1] < h:
                df.at[idx, 'max'] = h
                if last_max_idx_diff > 1:
                    last_max2 = last_max
                    last_max = [idx, h]
                    slice_low = df.loc[last_max2[0]: last_max[0], 'low']
                    new_min = slice_low.min()
                    new_min_idx = slice_low.idxmin()
                    last_min2 = last_min
                    last_min = [new_min_idx, new_min]
                    df.at[idx, 'min'] = new_min
                else:
                    last_max = [idx, h]
            if last_min[1] > l:
                df.at[idx, 'min'] = l
                if last_min_idx_diff > 1:
                    last_min2 = last_min
                    last_min = [idx, l]
                    slice_high = df.loc[last_min2[0]: last_min[0], 'high']
                    new_max = slice_high.max()
                    new_max_idx = slice_high.idxmax()
                    last_max2 = last_max
                    last_max = [new_max_idx, new_max]
                    df.at[idx, 'max'] = new_max
                else:
                    last_min = [idx, l]
        result = pd.DataFrame({'swing_high': df['max'].values,
                               'swing_low':  df['min'].values},
                              index=ohlc_df.index)
        return result

    def run(self,
            price_df: pd.DataFrame) -> pd.DataFrame:
        """
        Generate a combined DataFrame of indicators using two granularities.

        Steps:
        1. Resample price_df to `small_granularity` for MACD.
        2. Resample price_df to `large_granularity` for swings.
        3. Compute MACD & ATR-based stats on small candles.
        4. Compute swing highs/lows on large candles, forward-fill to small periods.
        5. Combine small OHLC, MACD stats, and swing levels into one DataFrame.
        """
        # Generate candles
        small_ohlc = self.generate_candles(price_df, self.small_granularity).dropna()
        large_ohlc = self.generate_candles(price_df, self.large_granularity).dropna()

        # Compute MACD on small candles
        macd_df = self.calculate_MACD(
            large_ohlc.reset_index().rename(columns={'index':'datetime'}),
            se=self.se, le=self.le, cont=self.cont)

        # Compute swings on large candles
        swings = self.detect_swing_points(large_ohlc)
        swings_small = swings.reindex(small_ohlc.index, method='ffill')

        # Combine: small OHLC, MACD stats, swings
        combined = small_ohlc.copy()
        combined = combined.join(macd_df)
        combined = combined.join(swings_small)

        return combined

In [ ]:
import pandas as pd
import numpy as np

class MultiGranularityIndicator:
    """
    Builds indicators on multiple candle granularities: MACD on a finer granularity,
    and swing highs/lows on a coarser granularity, then combines into one DataFrame.
    Adds SMA of length `se` on small candles and upper/lower bands at ±2×ATR (span `se`).
    """
    def __init__(self,
                 small_granularity: str = '5T',
                 large_granularity: str = '1H',
                 se: int = 12,
                 le: int = 26,
                 cont: int = 9):
        self.small_granularity = small_granularity
        self.large_granularity = large_granularity
        self.se = se
        self.le = le
        self.cont = cont

    @staticmethod
    def generate_candles(df: pd.DataFrame,
                         granularity: str) -> pd.DataFrame:
        """
        Resample a price series into OHLC candles.
        Expects a time-indexed DataFrame or Series named 'price'.
        Returns columns ['open','high','low','close'].
        """
        if isinstance(df, pd.Series):
            df = df.to_frame(name='price')
        ohlc = df.resample(granularity).agg({'price': ['first','max','min','last']})
        ohlc.columns = ['open','high','low','close']
        return ohlc

    @staticmethod
    def calculate_MACD(df_lag: pd.DataFrame,
                       price_col: str = 'close',
                       high_col: str = 'high',
                       low_col: str = 'low',
                       se: int = 12,
                       le: int = 26,
                       cont: int = 9) -> pd.DataFrame:
        """
        Compute ATR, MACD, ATR-normalized MACD, and 100-period percentile ranks.
        df_lag must have columns ['datetime', high_col, low_col, price_col].
        Returns DataFrame indexed by datetime with MACD & ATR stats.
        """
        df = df_lag.copy()
        if 'datetime' not in df.columns:
            df = df.reset_index().rename(columns={'index':'datetime'})
        df = df.dropna(subset=[price_col, high_col, low_col])

        # True Range & ATR
        hl = df[high_col] - df[low_col]
        hc = (df[high_col] - df[price_col].shift()).abs()
        lc = (df[low_col] - df[price_col].shift()).abs()
        tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/se, adjust=False, min_periods=se).mean()

        # MACD
        close = df[price_col]
        short_ema = close.ewm(span=se, adjust=False).mean()
        long_ema  = close.ewm(span=le, adjust=False).mean()
        macd      = short_ema - long_ema
        signal    = macd.ewm(span=cont, adjust=False).mean()
        hist      = macd - signal

        # Normalize
        macd_norm = macd / atr
        hist_norm = hist / atr

        # Percentile ranks
        macd_pr = macd_norm.rolling(window=100, min_periods=100)
        macd_pr = macd_pr.apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)
        hist_pr = hist_norm.rolling(window=100, min_periods=100)
        hist_pr = hist_pr.apply(lambda x: (x <= x.iloc[-1]).sum() / len(x) * 100, raw=False)

        out = pd.DataFrame({
            'ATR':               atr,
            'MACD':              macd,
            'Signal':            signal,
            'MACD_hist':         hist,
            'MACD_norm':         macd_norm,
            'MACD_hist_norm':    hist_norm,
            'MACD_norm_pr':      macd_pr,
            'MACD_hist_norm_pr': hist_pr
        })
        out.index = pd.to_datetime(df['datetime'])
        out.index.name = 'datetime'
        return out

    @staticmethod
    def detect_swing_points(ohlc_df: pd.DataFrame,
                            high_col: str = 'high',
                            low_col: str = 'low') -> pd.DataFrame:
        """
        Compute swing highs/lows based on the posted algorithm. Marks new extremes and recalculates opposite swing.
        Returns DataFrame with 'swing_high' and 'swing_low' columns.
        """
        df = ohlc_df.copy().reset_index()
        df['min'] = np.nan
        df['max'] = np.nan
        for idx, row in df.iterrows():
            o, h, l, c = row['open'], row['high'], row['low'], row['close']
            if idx == 0:
                last_min = [idx, l]
                last_min2 = [idx, l]
                last_max = [idx, h]
                last_max2 = [idx, h]
                df.at[idx, 'min'] = l
                df.at[idx, 'max'] = h
                continue
            last_max_idx_diff = idx - last_max[0]
            last_min_idx_diff = idx - last_min[0]
            if last_max[1] < h:
                df.at[idx, 'max'] = h
                if last_max_idx_diff > 1:
                    last_max2 = last_max
                    last_max = [idx, h]
                    slice_low = df.loc[last_max2[0]: last_max[0], 'low']
                    new_min = slice_low.min()
                    new_min_idx = slice_low.idxmin()
                    last_min2 = last_min
                    last_min = [new_min_idx, new_min]
                    df.at[idx, 'min'] = new_min
                else:
                    last_max = [idx, h]
            if last_min[1] > l:
                df.at[idx, 'min'] = l
                if last_min_idx_diff > 1:
                    last_min2 = last_min
                    last_min = [idx, l]
                    slice_high = df.loc[last_min2[0]: last_min[0], 'high']
                    new_max = slice_high.max()
                    new_max_idx = slice_high.idxmax()
                    last_max2 = last_max
                    last_max = [new_max_idx, new_max]
                    df.at[idx, 'max'] = new_max
                else:
                    last_min = [idx, l]
        result = pd.DataFrame({'swing_high': df['max'].values,
                               'swing_low':  df['min'].values},
                              index=ohlc_df.index)
        return result

    def run(self,
            price_df: pd.DataFrame) -> pd.DataFrame:
        """
        Generate a combined DataFrame of indicators using two granularities.

        Steps:
        1. Resample price_df to `small_granularity` for MACD and SMA.
        2. Resample price_df to `large_granularity` for swings and ATR.
        3. Compute MACD & ATR-based stats on small candles.
        4. Compute swing highs/lows on large candles, forward-fill to small periods.
        5. Compute SMA of length `se` on small candles.
        6. Compute 2×ATR with span `se` on large candles, forward-fill to small periods.
        7. Calculate upper and lower bands: SMA ± 2×ATR.
        8. Combine all into one DataFrame.
        """
        # Generate candles
        small_ohlc = self.generate_candles(price_df, self.small_granularity).dropna()
        large_ohlc = self.generate_candles(price_df, self.large_granularity).dropna()

        # Compute MACD on small candles
        macd_df = self.calculate_MACD(
            large_ohlc.reset_index().rename(columns={'index':'datetime'}),
            se=self.se, le=self.le, cont=self.cont)

        # Detect swings on large candles
        swings = self.detect_swing_points(large_ohlc)
        swings_small = swings.reindex(small_ohlc.index, method='ffill')

        # Simple Moving Average on small candles
        sma_small = small_ohlc['close'].rolling(window=self.se, min_periods=self.se).mean()
        small_ohlc['SMA_se'] = sma_small

        # ATR on large candles with span se
        high = large_ohlc['high']
        low  = large_ohlc['low']
        close= large_ohlc['close']
        hl = high - low
        hc = (high - close.shift()).abs()
        lc = (low - close.shift()).abs()
        tr_large = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        atr_large = tr_large.ewm(alpha=1/self.se, adjust=False, min_periods=self.se).mean()
        two_atr_large = 2 * atr_large
        # Forward-fill to small index
        two_atr_small = two_atr_large.reindex(small_ohlc.index, method='ffill')
        small_ohlc['two_ATR_se'] = two_atr_small

        # Upper and Lower Bands around SMA
        small_ohlc['upper_band'] = small_ohlc['SMA_se'] + small_ohlc['two_ATR_se']
        small_ohlc['lower_band'] = small_ohlc['SMA_se'] - small_ohlc['two_ATR_se']

        # Combine everything
        combined = small_ohlc.copy()
        combined = combined.join(macd_df)
        combined = combined.join(swings_small)

        return combined


In [ ]:
def plot_with_levels(combined: pd.DataFrame,
                     upper_col: str = 'MACD_norm',
                     lower_col: str = 'MACD_hist_norm',
                     level_cols: tuple = ('swing_high', 'swing_low')):
    # confirm GUI backend
    import matplotlib as mpl
    print("Backend:", mpl.get_backend())

    df = combined.dropna(subset=['open','high','low','close'])
    n = len(df)
    x = np.arange(n)
    width = 0.6

    fig, (ax1, ax2, ax3) = plt.subplots(
        nrows=3, ncols=1, sharex=True, figsize=(12, 10),
        gridspec_kw={'height_ratios': [3, 1, 1]}
    )

    # --- Price candles + swings on ax1 ---
    for i, (_, row) in enumerate(df.iterrows()):
        o, h, l, c = row['open'], row['high'], row['low'], row['close']
        ax1.vlines(i, l, h, color='black', linewidth=1)
        ax1.bar(i, c-o, width, bottom=o,
                color='green' if c > o else 'red', edgecolor='black')
    ax1.plot(x, df[level_cols[0]], '.', ms=8, color='blue',  label=level_cols[0])
    ax1.plot(x, df[level_cols[1]], '.', ms=8, color='purple', label=level_cols[1])
    # ax1.plot(x, df['SMA_se'])
    # ax1.plot(x, df['lower_band'])
    # ax1.plot(x, df['upper_band'])
    ax1.set_title('Price Candles with Swing Levels')
    ax1.set_ylabel('Price')
    ax1.legend(loc='upper left')
    pad = (df['high'].max() - df['low'].min()) * 0.05
    ax1.set_ylim(df['low'].min() - pad, df['high'].max() + pad)
    ax1.grid(True)

    # --- MACD normalized on ax2 ---
    ax2.plot(x, df[upper_col], '-o', label=upper_col)
    ax2.set_ylabel(upper_col)
    ax2.legend(loc='upper left')
    ax2.grid(True)

    # --- MACD histogram on ax3 ---
    ax3.plot(x, df[lower_col], '-o', label=lower_col)
    ax3.set_ylabel(lower_col)
    ax3.legend(loc='upper left')
    ax3.grid(True)
    ax3.set_xlabel('Date')

    # format x-ticks as datetime
    N = max(n // 10, 1)
    ticks  = x[::N]
    labels = df.index.strftime('%Y-%m-%d %H:%M')[::N]
    ax3.set_xticks(ticks)
    ax3.set_xticklabels(labels, rotation=45, ha='right')

    # custom hover readout on main chart
    def fmt_coord(xi, y):
        idx = int(round(xi))
        if 0 <= idx < len(df):
            return f"{df.index[idx]:%Y-%m-%d %H:%M}, {y:.2f}"
        return f"{y:.2f}"
    ax1.format_coord = fmt_coord

    # --- full crosshair across all axes, with blit turned off ---
    # store directly on the figure so it's "used"
    fig._multi = MultiCursor(
        fig.canvas,
        (ax1, ax2, ax3),
        useblit=False,
        horizOn=False, vertOn=True,
        color='gray', lw=1
    )

    fig.canvas.draw()
    plt.tight_layout()
    plt.show()         # blocks until window is closed  # give a moment for the cursor to render


C:\Users\krajcovic\AppData\Local\Temp\ipykernel_20056\3024722400.py:32: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.

C:\Users\krajcovic\AppData\Local\Temp\ipykernel_20056\3024722400.py:32: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [ ]:
combined_df.head()

In [ ]:
combined_df.describe()

In [ ]:
import matplotlib.pyplot as plt
combined_df[['open','close','MACD','MACD_norm']]\
    .plot(subplots=True, layout=(2,2), figsize=(10,8))
plt.tight_layout()
plt.show()